# A Python programmer's tour of MeTTa

Nothing here is written as MeTTa source. `S` makes a symbol, `V` makes a variable, and calling a symbol builds an expression, so `S.Parent(S.Tom, S.Bob)` IS the atom `(Parent Tom Bob)` before any engine sees it. A built term is knowledge already; a string has to be parsed before it is anything, and it is opaque to your editor, your type checker and `grep` until then.

`space()` starts an isolated engine-backed space for the rest of the tour.

In [1]:
from metta import S, V, equation, fn, space

m = space()
parent = S.Parent(S.Tom, S.Bob)
person = V.person
parent, person, tuple(parent)

((Parent Tom Bob), $person, (Parent, Tom, Bob))

## Put a small family graph in a space

A space is a collection, and `+=` means what it means for a `set`. Three facts go in as three terms.

Then ask. `m.match(pattern)` takes the pattern you built and answers rows keyed by the pattern's own variable names, so `row.parent` reads back the name you wrote. Leave the answer as a cell's last expression and Jupyter renders its columns as a table.

In [2]:
for older, younger in (("Tom", "Bob"), ("Bob", "Ann"), ("Ann", "Zoe")):
    m += S.Parent(S[older], S[younger])

len(m)

3

In [3]:
m.match(S.Parent(V.parent, V.child))

parent,child
Tom,Bob
Bob,Ann
Ann,Zoe


## Keep Python and MeTTa in one session

Load `metta.ipython` in an ordinary Python kernel to register `%%metta`. The extension owns a default runtime, and `use(m)` points the magic at the space Python already created. A `%%metta` cell is the one place MeTTa source belongs in this notebook: text is exactly what it is for.

In [4]:
%load_ext metta.ipython
from metta.ipython import use

use(m)

In [5]:
%%metta
!(+ 1 2)
(Parent Zoe Lia)
!(match (context-space) (Parent $parent $child) ($parent $child))

3
(Tom Bob) (Bob Ann) (Ann Zoe) (Zoe Lia)


[[Grounded(3)], [(Tom Bob), (Bob Ann), (Ann Zoe), (Zoe Lia)]]

In [6]:
m.match(S.Parent(V.parent, V.child))

parent,child
Tom,Bob
Bob,Ann
Ann,Zoe
Zoe,Lia


## Let MeTTa call a Python function

`@m.op` registers a Python callable as a MeTTa operation. Every registration declares its strongest observable effect; this formatter is `pureStructural`. Type annotations become a MeTTa declaration. The Python name reaches MeTTa through Python's own casing, so `describe_link` installs `describe-link`, and `S["describe-link"]` is how you name it back. `.one()` is the cardinality door: exactly one answer, or a refusal that says how many there were.

In [7]:
@m.op(effect="pureStructural")
def describe_link(parent_name: str, child_name: str) -> str:
    return f"{parent_name} is parent of {child_name}"


m.answers(S["describe-link"]("Tom", "Bob")).one()

'Tom is parent of Bob'

## Compile a Python function into MeTTa equations

`@m.define` reads the function's own body and installs the equation it means. `.source()` shows what was stored.

The compiled function is then callable from Python, and what comes back is a LIST: a MeTTa function answers a bag, and one answer is a bag of one rather than a special case. `.py` keeps the ordinary Python function beside it, which is how you compare the two.

In [8]:
@m.define
def generation_score(levels):
    if levels == 0:
        return 0
    return levels + generation_score(levels - 1)


generation_score.source()

'(= (generation-score $levels) (if (py-eq $levels 0) 0 (+ $levels (generation-score (- $levels 1)))))'

In [9]:
generation_score(3), generation_score.py(3)

([Grounded(6)], 6)

## Check a MeTTa type from Python

A type declaration is an atom like any other: `(: Tom Person)` is `S[":"](S.Tom, S.Person)`. `m.cast` then asks the engine to CHECK the claim. A successful symbolic cast answers the same symbol; a value the type does not admit raises `CastError` rather than returning an approximation.

In [10]:
m += S[":"](S.Tom, S.Person)
m.cast(S.Tom, S.Person)

Tom

## Read a reduction as an indented story

`m.trace` takes the same term `answers` and `eval` take. Each call enters and each answer returns; builtins stay out, so the indentation follows the MeTTa functions in the program.

In [11]:
for event in m.trace(S["generation-score"](3)):
    print(event)

-> (generation-score 3)
  -> (generation-score 2)
    -> (generation-score 1)
      -> (generation-score 0)
      (generation-score 0) = 0
    (generation-score 1) = 1
  (generation-score 2) = 3
(generation-score 3) = 6


## Find mistakes that would otherwise stay inert

MeTTa leaves many bad calls unreduced rather than raising. Put two deliberate mistakes in a scratch space and ask it. `ghost-fn` is declared and never defined; `one-arg` is called with two arguments. Each finding names the problem and its subject.

In [12]:
sloppy = space()
sloppy += S[":"](S["ghost-fn"], S["->"](S.Number, S.Number))
sloppy += equation(S["one-arg"](V.x)).to(V.x)
sloppy += equation(S["bad-caller"]()).to(S["one-arg"](1, 2))

for finding in sloppy.lint():
    print(finding)

[declared-but-undefined] ghost-fn: declared (-> Number Number) but nothing defines it; every call will stay unreduced
[arity-mismatch] one-arg: called with 2 argument(s) but defined for [1]


## Ask why an answer holds

Two equations for one name, both of which apply: a parent is an ancestor, and so is an ancestor's ancestor. `equation(head).to(body)` installs one, and `fn.match` BUILDS the match rather than running it, which is the difference between writing a rule and asking a question.

Asking for Tom's ancestors then walks both equations, and the answers are a bag, so the cell lists them.

In [13]:
m += equation(S.ancestor(V.x)).to(fn.match(m, S.Parent(V.x, V.child), V.child))
m += equation(S.ancestor(V.x)).to(
    S.ancestor(fn.match(m, S.Parent(V.x, V.middle), V.middle))
)

m.answers(S.ancestor(S.Tom))

[Bob, Ann, Zoe, Lia]

`m.derivation(...)` answers every proof of every answer, so pick the one you want to read. Ann is two generations down, which is the answer only the recursive equation reaches: its proof names that equation, the fact it matched, and the shorter proof it stood on. Leave a proof as the cell value and Jupyter renders it as a tree.

In [14]:
(proof,) = [p for p in m.derivation(S.ancestor(S.Tom)) if p.answer == S.Ann]
proof

Derivation(call=(ancestor Tom), answer=Ann, children=(Step(call=(ancestor Tom), answer=Ann, equation=(=
  (ancestor $_2058)
  (ancestor (match &pyspace_1 (Parent $_2058 $_2112) $_2112))), children=(Fact(space='&pyspace_1', atom=(Parent Tom Bob)), Step(call=(ancestor Bob), answer=Ann, equation=(= (ancestor $_2964) (match &pyspace_1 (Parent $_2964 $_3006) $_3006)), children=(Fact(space='&pyspace_1', atom=(Parent Bob Ann)),)))),))